In [ ]:
import re
import pandas as pd
from collections import Counter
import spacy
import emoji

# =========================================================
# LOAD DATASETS
# =========================================================

germeval_raw = pd.read_csv(
    "https://raw.githubusercontent.com/happy522/NE-Masking-for-DeBiasing-Text-Classification/refs/heads/main/Dataset/germeval2018.csv"
)

hasoc_raw = pd.read_csv(
    "https://raw.githubusercontent.com/happy522/NE-Masking-for-DeBiasing-Text-Classification/refs/heads/main/Dataset/HASOC.csv"
)

gahd_raw = pd.read_csv(
    "https://raw.githubusercontent.com/happy522/NE-Masking-for-DeBiasing-Text-Classification/refs/heads/main/Dataset/GAHD.csv"
)

# =========================================================
# INSTALL
# pip install pandas spacy emoji
# python -m spacy download de_core_news_lg
# =========================================================

nlp = spacy.load("de_core_news_lg")

# =========================================================
# FIND TEXT COLUMN
# =========================================================

def find_text_column(df):

    possible_cols = [
        "text",
        "tweet",
        "content",
        "sentence",
        "comment"
    ]

    for col in possible_cols:
        if col in df.columns:
            return col

    for col in df.columns:
        if df[col].dtype == "object":
            return col

    raise ValueError("No text column found")


# =========================================================
# CLEAN TEXT
# =========================================================

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove hashtags but keep word
    # #Berlin -> Berlin
    text = re.sub(r"#(\w+)", r"\1", text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace="")

    # Remove special characters
    text = re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================================================
# SAVE ENTITY FILE
# =========================================================

def save_entity_file(entity_list, filename):

    cleaned_entities = []

    for e in entity_list:

        e = e.strip()

        if len(e) > 1:
            cleaned_entities.append(e)

    # Count frequency
    counter = Counter(cleaned_entities)

    out_df = pd.DataFrame({
        "entity": list(counter.keys()),
        "count": list(counter.values())
    })

    # Sort by count
    out_df = out_df.sort_values(
        by="count",
        ascending=False
    )

    # Remove duplicate entities ignoring case
    out_df["lower"] = out_df["entity"].str.lower()

    out_df = out_df.drop_duplicates(
        subset=["lower"]
    )

    out_df = out_df.drop(columns=["lower"])

    # Save CSV
    out_df.to_csv(filename, index=False)

    print(f"Saved: {filename}")

    return out_df


# =========================================================
# EXTRACT ENTITIES
# =========================================================

def extract_entities(df, dataset_name):

    text_col = find_text_column(df)

    print(f"\nProcessing {dataset_name}")
    print(f"Using text column: {text_col}")

    texts = (
        df[text_col]
        .dropna()
        .astype(str)
        .apply(clean_text)
        .tolist()
    )

    entity_dict = {
        "PER": [],
        "ORG": [],
        "LOC": []
    }

    # Batch processing
    for doc in nlp.pipe(texts, batch_size=32):

        for ent in doc.ents:

            label = ent.label_

            if label in entity_dict:

                entity = ent.text.strip()

                if len(entity) > 1:
                    entity_dict[label].append(entity)

    return entity_dict


# =========================================================
# PROCESS DATASETS
# =========================================================

datasets = {
    "Germeval": germeval_raw,
    "HASOC": hasoc_raw,
    "GAHD": gahd_raw
}

# merged storage
merged_entities = {
    "PER": [],
    "ORG": [],
    "LOC": []
}

# =========================================================
# INDIVIDUAL DATASET FILES
# =========================================================

for dataset_name, df in datasets.items():

    entities = extract_entities(df, dataset_name)

    # Save individual files
    for ent_type, ent_list in entities.items():

        filename = f"{dataset_name}_{ent_type}.csv"

        save_entity_file(ent_list, filename)

        # add to merged storage
        merged_entities[ent_type].extend(ent_list)


# =========================================================
# SAVE MERGED FILES
# =========================================================

print("\nSaving merged files...")

for ent_type, entity_list in merged_entities.items():

    filename = f"merged_{ent_type}.csv"

    save_entity_file(entity_list, filename)


# =========================================================
# OUTPUT FILES
# =========================================================

# Individual files:
# Germeval_PER.csv
# Germeval_ORG.csv
# Germeval_LOC.csv
#
# HASOC_PER.csv
# HASOC_ORG.csv
# HASOC_LOC.csv
#
# GAHD_PER.csv
# GAHD_ORG.csv
# GAHD_LOC.csv
#
# Merged files:
# merged_PER.csv
# merged_ORG.csv
# merged_LOC.csv
# =========================================================

In [ ]:


# =========================================================
# CREATE ENTITY SETS
# =========================================================

entity_sets = {}

for dataset_name, entity_data in datasets.items():

    entity_sets[dataset_name] = {}

    for ent_type, df in entity_data.items():

        entity_sets[dataset_name][ent_type] = set(
            df["entity"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

# =========================================================
# GRAPH 1:
# UNIQUE ENTITY COUNTS
# =========================================================

for ent_type in entity_types:

    counts = []

    dataset_names = []

    for dataset_name in datasets:

        count = len(entity_sets[dataset_name][ent_type])

        counts.append(count)

        dataset_names.append(dataset_name)

    plt.figure(figsize=(8, 5))

    plt.bar(dataset_names, counts)

    plt.title(f"Unique {ent_type} Entities per Dataset")

    plt.xlabel("Dataset")

    plt.ylabel("Number of Unique Entities")

    plt.tight_layout()

    plt.savefig(f"graph_unique_{ent_type}.png")

    plt.close()

    print(f"Saved: graph_unique_{ent_type}.png")


# =========================================================
# GRAPH 2:
# JACCARD SIMILARITY HEATMAP STYLE
# =========================================================

for ent_type in entity_types:

    similarity_matrix = []

    dataset_names = list(datasets.keys())

    for d1 in dataset_names:

        row = []

        for d2 in dataset_names:

            set1 = entity_sets[d1][ent_type]

            set2 = entity_sets[d2][ent_type]

            similarity = len(set1 & set2) / len(set1 | set2)

            row.append(similarity)

        similarity_matrix.append(row)

    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=dataset_names,
        columns=dataset_names
    )

    plt.figure(figsize=(6, 5))

    plt.imshow(similarity_df.values)

    plt.colorbar(label="Jaccard Similarity")

    plt.xticks(range(len(dataset_names)), dataset_names)

    plt.yticks(range(len(dataset_names)), dataset_names)

    plt.title(f"{ent_type} Entity Similarity")

    plt.tight_layout()

    plt.savefig(f"graph_similarity_{ent_type}.png")

    plt.close()

    print(f"Saved: graph_similarity_{ent_type}.png")


# =========================================================
# GRAPH 3:
# COMMON ENTITIES BETWEEN DATASETS
# =========================================================

for ent_type in entity_types:

    pair_names = []

    overlap_counts = []

    for d1, d2 in combinations(datasets.keys(), 2):

        common = (
            entity_sets[d1][ent_type]
            .intersection(entity_sets[d2][ent_type])
        )

        pair_names.append(f"{d1}\nvs\n{d2}")

        overlap_counts.append(len(common))

    plt.figure(figsize=(8, 5))

    plt.bar(pair_names, overlap_counts)

    plt.title(f"Common {ent_type} Entities Between Datasets")

    plt.xlabel("Dataset Pair")

    plt.ylabel("Number of Common Entities")

    plt.tight_layout()

    plt.savefig(f"graph_overlap_{ent_type}.png")

    plt.close()

    print(f"Saved: graph_overlap_{ent_type}.png")


# =========================================================
# GRAPH 4:
# TOP 15 ENTITIES FREQUENCY
# =========================================================

for dataset_name in datasets:

    for ent_type in entity_types:

        df = datasets[dataset_name][ent_type]

        top_df = df.head(15)

        plt.figure(figsize=(10, 6))

        plt.barh(
            top_df["entity"],
            top_df["count"]
        )

        plt.title(
            f"Top 15 {ent_type} Entities - {dataset_name}"
        )

        plt.xlabel("Frequency")

        plt.ylabel("Entity")

        plt.tight_layout()

        plt.savefig(
            f"graph_top15_{dataset_name}_{ent_type}.png"
        )

        plt.close()

        print(
            f"Saved: graph_top15_{dataset_name}_{ent_type}.png"
        )


# =========================================================
# GENERATED FILES
# =========================================================

# graph_unique_PER.png
# graph_unique_ORG.png
# graph_unique_LOC.png
#
# graph_similarity_PER.png
# graph_similarity_ORG.png
# graph_similarity_LOC.png
#
# graph_overlap_PER.png
# graph_overlap_ORG.png
# graph_overlap_LOC.png
#
# graph_top15_GAHD_PER.png
# ...
# =========================================================

In [ ]:

# =========================================================
# GRAPH 1:
# 3-WAY VENN DIAGRAMS
# =========================================================

for ent_type in entity_types:

    plt.figure(figsize=(8, 8))

    venn3(
        [
            entity_sets["GAHD"][ent_type],
            entity_sets["HASOC"][ent_type],
            entity_sets["Germeval"][ent_type]
        ],
        set_labels=("GAHD", "HASOC", "Germeval")
    )

    plt.title(f"3-Way Overlap of {ent_type} Entities")

    plt.savefig(f"venn_{ent_type}.png")

    plt.close()

    print(f"Saved: venn_{ent_type}.png")


# =========================================================
# GRAPH 2:
# ENTITIES PRESENT IN ALL 3 DATASETS
# =========================================================

for ent_type in entity_types:

    common_all = (
        entity_sets["GAHD"][ent_type]
        .intersection(entity_sets["HASOC"][ent_type])
        .intersection(entity_sets["Germeval"][ent_type])
    )

    common_freq = []

    for entity in common_all:

        total_freq = 0

        for dataset_name in datasets:

            df = datasets[dataset_name][ent_type]

            row = df[
                df["entity"].str.lower() == entity
            ]

            if not row.empty:
                total_freq += int(row.iloc[0]["count"])

        common_freq.append((entity, total_freq))

    # top common entities
    common_freq = sorted(
        common_freq,
        key=lambda x: x[1],
        reverse=True
    )[:15]

    if len(common_freq) == 0:
        continue

    entities = [x[0] for x in common_freq]
    counts = [x[1] for x in common_freq]

    plt.figure(figsize=(12, 7))

    plt.barh(entities, counts)

    plt.title(
        f"Top Common {ent_type} Entities Across ALL Datasets"
    )

    plt.xlabel("Combined Frequency")

    plt.ylabel("Entity")

    plt.tight_layout()

    plt.savefig(f"top_common_all_{ent_type}.png")

    plt.close()

    print(f"Saved: top_common_all_{ent_type}.png")


# =========================================================
# GRAPH 3:
# UNIQUE VS SHARED ENTITIES
# =========================================================

for ent_type in entity_types:

    gahd = entity_sets["GAHD"][ent_type]
    hasoc = entity_sets["HASOC"][ent_type]
    germeval = entity_sets["Germeval"][ent_type]

    unique_gahd = len(gahd - hasoc - germeval)
    unique_hasoc = len(hasoc - gahd - germeval)
    unique_germeval = len(germeval - gahd - hasoc)

    shared_all = len(gahd & hasoc & germeval)

    shared_partial = len(
        (
            (gahd & hasoc)
            | (gahd & germeval)
            | (hasoc & germeval)
        ) - (gahd & hasoc & germeval)
    )

    labels = [
        "Unique\nGAHD",
        "Unique\nHASOC",
        "Unique\nGermeval",
        "Shared\n2 Datasets",
        "Shared\nAll 3"
    ]

    values = [
        unique_gahd,
        unique_hasoc,
        unique_germeval,
        shared_partial,
        shared_all
    ]

    plt.figure(figsize=(10, 6))

    plt.bar(labels, values)

    plt.title(
        f"Unique vs Shared {ent_type} Entities"
    )

    plt.ylabel("Entity Count")

    plt.tight_layout()

    plt.savefig(f"shared_vs_unique_{ent_type}.png")

    plt.close()

    print(f"Saved: shared_vs_unique_{ent_type}.png")


# =========================================================
# GRAPH 4:
# TOP ENTITIES ACROSS ALL DATASETS
# =========================================================

for ent_type in entity_types:

    global_counter = Counter()

    for dataset_name in datasets:

        df = datasets[dataset_name][ent_type]

        for _, row in df.iterrows():

            entity = str(row["entity"]).lower()

            count = int(row["count"])

            global_counter[entity] += count

    top_entities = global_counter.most_common(20)

    entities = [x[0] for x in top_entities]
    counts = [x[1] for x in top_entities]

    plt.figure(figsize=(12, 8))

    plt.barh(entities, counts)

    plt.title(
        f"Top 20 {ent_type} Entities Across ALL Datasets"
    )

    plt.xlabel("Total Frequency")

    plt.ylabel("Entity")

    plt.tight_layout()

    plt.savefig(f"global_top20_{ent_type}.png")

    plt.close()

    print(f"Saved: global_top20_{ent_type}.png")


# =========================================================
# GRAPH 5:
# DATASET CONTRIBUTION TO COMMON ENTITIES
# =========================================================

for ent_type in entity_types:

    common_all = (
        entity_sets["GAHD"][ent_type]
        .intersection(entity_sets["HASOC"][ent_type])
        .intersection(entity_sets["Germeval"][ent_type])
    )

    contributions = {
        "GAHD": 0,
        "HASOC": 0,
        "Germeval": 0
    }

    for dataset_name in datasets:

        df = datasets[dataset_name][ent_type]

        common_df = df[
            df["entity"].str.lower().isin(common_all)
        ]

        contributions[dataset_name] = common_df["count"].sum()

    plt.figure(figsize=(8, 6))

    plt.pie(
        list(contributions.values()),
        labels=list(contributions.keys()),
        autopct='%1.1f%%'
    )

    plt.title(
        f"Contribution to Shared {ent_type} Entities"
    )

    plt.savefig(f"shared_contribution_{ent_type}.png")

    plt.close()

    print(f"Saved: shared_contribution_{ent_type}.png")


# =========================================================
# GENERATED FILES
# =========================================================

# venn_PER.png
# venn_ORG.png
# venn_LOC.png
#
# top_common_all_PER.png
# top_common_all_ORG.png
# top_common_all_LOC.png
#
# shared_vs_unique_PER.png
# shared_vs_unique_ORG.png
# shared_vs_unique_LOC.png
#
# global_top20_PER.png
# global_top20_ORG.png
# global_top20_LOC.png
#
# shared_contribution_PER.png
# shared_contribution_ORG.png
# shared_contribution_LOC.png
# =========================================================